In [48]:
import pandas as pd
import numpy as np
import nltk
import warnings
warnings.filterwarnings('ignore')

In [49]:
df = pd.read_csv('amazon_india_products.csv')
df = df[['Title', 'Product Description', 'Category']]
df

,Title,Product Description,Category
0,Sparco Premium Printers - Model-6476,Backed by a manufacturer warranty and availabl...,Office Supplies
1,Parle Premium Snacks (Rose Gold),Customer care support available in major India...,Grocery & Gourmet Foods
2,CarObot Durable Car Chargers (Silver) - Model-...,Environmentally responsible packaging aligns w...,Automotive
3,SKY EC Smart Pet Carriers - Model-1258,Ideal for both personal and professional use a...,Pet Supplies
4,Columbia Portable Cycling Helmets - Model-2302,Environmentally responsible packaging aligns w...,Sports & Outdoors
...,...,...,...
8995,Himalaya Compact Pulse Oximeters (Navy Blue) -...,Comes with a user manual in Hindi and English ...,Health & Wellness
8996,Michelin Ergonomic Car Mats - Model-3381,The Michelin brand has been trusted by million...,Automotive
8997,Acer Ergonomic Cameras (Blue) - Model-6105,"Lightweight and easy to carry, making it perfe...",Electronics
8998,MDH Eco-Friendly Juices (Grey),"Introducing the MDH Eco-Friendly Juices, desig...",Grocery & Gourmet Foods


In [50]:
df.isnull().sum()

,0
Title,0
Product Description,0
Category,0


In [51]:
from nltk.stem.snowball import SnowballStemmer
stemmer = SnowballStemmer('english')

def tokenize_stem(text):
  tokens = nltk.word_tokenize(text.lower())            #Tokenization
  stemmed = [stemmer.stem(token) for token in tokens]   #Stemming words into base form
  return " ".join(stemmed)

In [52]:
nltk.download('punkt_tab')

df['stemmed_tokens'] = df.apply(lambda x: tokenize_stem(x['Title']+ " " + x['Product Description']), axis=1)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [53]:
df.head()

,Title,Product Description,Category,stemmed_tokens
0,Sparco Premium Printers - Model-6476,Backed by a manufacturer warranty and availabl...,Office Supplies,sparco premium printer - model-6476 back by a ...
1,Parle Premium Snacks (Rose Gold),Customer care support available in major India...,Grocery & Gourmet Foods,parl premium snack ( rose gold ) custom care s...
2,CarObot Durable Car Chargers (Silver) - Model-...,Environmentally responsible packaging aligns w...,Automotive,carobot durabl car charger ( silver ) - model-...
3,SKY EC Smart Pet Carriers - Model-1258,Ideal for both personal and professional use a...,Pet Supplies,sky ec smart pet carrier - model-1258 ideal fo...
4,Columbia Portable Cycling Helmets - Model-2302,Environmentally responsible packaging aligns w...,Sports & Outdoors,columbia portabl cycl helmet - model-2302 envi...


In [57]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Initialize the TF-IDF Vectorizer with the custom tokenizer
tfidfv = TfidfVectorizer(tokenizer=tokenize_stem)

# Fit the vectorizer on the entire corpus of stemmed tokens once
tfidf_matrix = tfidfv.fit_transform(df['stemmed_tokens'])

In [58]:
def search_product(query):
  stemmed_query = tokenize_stem(query)
  # Transform the query using the *already fitted* tfidfv
  query_vec = tfidfv.transform([stemmed_query])

  # Calculate cosine similarity between the query vector and all document vectors in tfidf_matrix
  # This will result in a 1D array of similarity scores, one for each document
  similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()

  df['similarity'] = similarities
  # Sort in descending order to get the highest similarity first (most relevant)
  res = df.sort_values(by=['similarity'], ascending=False).head(10)[['Title', 'Product Description', 'Category']]
  return res

In [59]:
search_product('Sparco Premium Printers - Model-6476')

,Title,Product Description,Category
8576,MuscleTech Energy-Efficient Blood Pressure Mon...,Ideal for both personal and professional use a...,Health & Wellness
973,Fisher-Price Wireless Remote Control Cars - Mo...,Highly rated by verified buyers on Amazon Indi...,Toys & Games
269,Canon High-Performance Paper Shredders - Model...,Compatible with Indian power standards (220V/5...,Office Supplies
229,Casio Premium Cajon - Model-4626,This cajon features cutting-edge technology to...,Musical Instruments
98,Dabur Ultra-Slim Protein Powders (Gold) - Mode...,Comes with a user manual in Hindi and English ...,Health & Wellness
3460,HUFT Multi-Purpose Pet Carriers (Orange) - Mod...,The HUFT brand has been trusted by millions of...,Pet Supplies
5342,Skillofun Waterproof Puzzles - Model-7626,"Introducing the Skillofun Waterproof Puzzles, ...",Toys & Games
8610,Realme Noise-Cancelling Computer Monitors - Mo...,"The red finish gives it a sleek, modern appear...",Electronics
3936,Luxor Professional-Grade Tape Dispensers - Mod...,Introducing the Luxor Professional-Grade Tape ...,Office Supplies
2921,Puma Premium Sports Shoes (Navy Blue) - Model-...,Ideal for both personal and professional use a...,Fashion
